# Reuters RNN and LSTM experiments

Machine learning / artificial intelligence coursework, 2025. Recovered educational coursework adapted from course material.

**Status:** Legacy educational experiment; outputs removed and model training not rerun.

Original outputs, notebook metadata, and assignment prompts were removed for this public edition. See the topic README and PROVENANCE.json for scope and attribution. Static syntax review only; experiments have not been rerun.


In [ ]:
from keras.datasets import reuters
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, Embedding, SimpleRNN
import numpy as np
from keras.layers import Dropout


max_words = 10000

# Load the Reuters dataset
(x_train, y_train), (x_test, y_test) = reuters.load_data(num_words=max_words, test_split=0.2)

# Get the class names
class_names = ["cocoa","grain","veg-oil","earn","acq","wheat","copper","housing","money-supply",
               "coffee","sugar","trade","reserves","ship","cotton","carcass","crude","nat-gas",
               "cpi","money-fx","interest","gnp","meal-feed","alum","oilseed","gold","tin",
               "strategic-metal","livestock","retail","ipi","iron-steel","rubber","heat","jobs",
               "lei","bop","zinc","orange","pet-chem","dlr","gas","silver","wpi","hog","lead"]

# Define the word index
word_index = reuters.get_word_index()
word_index = {k: (v+3) for k, v in word_index.items()}
word_index['<PAD>'] = 0
word_index['<START>'] = 1
word_index['<UNK>'] = 2
word_index['<UNUSED>'] = 3

# Reverse word index
reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])

# Set the maximum sequence length
maxlen = 100

# Pad the sequences
x_train = pad_sequences(x_train, padding='post', maxlen=maxlen)
x_test = pad_sequences(x_test, padding='post', maxlen=maxlen)


In [ ]:
# One-hot encode the labels
num_classes = len(class_names)
y_train_one_hot = to_categorical(y_train, num_classes)
y_test_one_hot = to_categorical(y_test, num_classes)


In [ ]:
# Define and train the RNN model
from keras.layers import Dense, Dropout, Embedding, SimpleRNN
from keras.models import Sequential

# Define the SimpleRNN model
model = Sequential()
model.add(Embedding(max_words, 256))
model.add(SimpleRNN(256, activation='tanh', return_sequences=True))
model.add(Dropout(0.5))
model.add(SimpleRNN(256, activation='tanh'))
model.add(Dropout(0.5))
model.add(Dense(num_classes, activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(x_train, y_train_one_hot, epochs=10, batch_size=128, validation_split=0.1)


In [ ]:
# Print some samples from the test set along with the predicted label
for i in range(20):
    x_sample = x_test[i]
    y_true = np.argmax(y_test_one_hot[i])
    y_pred = np.argmax(model.predict(x_sample.reshape(1, maxlen))[0])
    sample_text = ' '.join([reverse_word_index.get(idx, '') for idx in x_sample])
    print(f'Test sample {i+1}: {sample_text}')
    print(f'True label: {class_names[y_true]}, Predicted label: {class_names[y_pred]} ({y_pred})\n')


In [ ]:
from sklearn.metrics import classification_report

# Get the predicted labels
y_pred = model.predict(x_test)

# Convert the predicted probabilities to labels
y_pred = np.argmax(y_pred, axis=1)

# Print the classification report
print(classification_report(y_test, y_pred, target_names=class_names))


In [ ]:
from keras.layers import LSTM

model = Sequential()
model.add(Embedding(max_words, 256))
model.add(LSTM(128, activation='tanh', return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(128, activation='tanh'))
model.add(Dropout(0.5))
model.add(Dense(num_classes, activation='softmax'))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train_one_hot, epochs=10, batch_size=128, validation_split=0.1)


In [ ]:
from sklearn.metrics import classification_report

# Get the predicted labels
y_pred = model.predict(x_test)

# Convert the predicted probabilities to labels
y_pred = np.argmax(y_pred, axis=1)

# Print the classification report
print(classification_report(y_test, y_pred, target_names=class_names))
